# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ayesha-Shahzadkhan/flyrank-assignment1/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [12]:
!git clone https://github.com/Ayesha-Shahzadkhan/flyrank-assignment1.git
%cd flyrank-assignment1

Cloning into 'flyrank-assignment1'...
remote: Enumerating objects: 148, done.
remote: Counting objects: 100% (148/148), done.
remote: Compressing objects: 100% (102/102), done.
remote: Total 148 (delta 58), reused 99 (delta 30), pack-reused 0 (from 0)
Receiving objects: 100% (148/148), 1.87 MiB | 10.39 MiB/s, done.
Resolving deltas: 100% (58/58), done.
/content/flyrank-assignment1/flyrank-assignment1


In [13]:
import os
print(os.path.exists("data/raw/content_refresh_anonymized.csv"))

True


In [14]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)
df.columns.tolist()

(30000, 44)


['content_id',
 'client_id',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'content_type',
 'main_intent',
 'word_count',
 'char_count',
 'provider_used',
 'model_used',
 'impressions_90d',
 'clicks_90d',
 'pageviews_90d',
 'sessions_90d',
 'users_90d',
 'engaged_sessions_90d',
 'ai_sessions_90d',
 'scroll_events_90d',
 'days_with_impressions',
 'days_with_sessions',
 'impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'days_since_last_update',
 'freshness_tier',
 'word_count_tier',
 'char_count_tier',
 'ctr',
 'avg_position',
 'engagement_rate',
 'scroll_rate',
 'ai_traffic_pct',
 'impression_tier',
 'position_tier',
 'trend_direction',
 'trend_pct']

In [15]:
df["trend_direction"].value_counts()

,count
trend_direction,
down,16262
stable,5962
up,4388
new,2236
flat,1152


In [16]:
df['days_since_last_update'].describe()

,days_since_last_update
count,30000.000000
mean,46.098300
std,42.078709
min,1.000000
25%,20.000000
50%,20.000000
75%,104.000000
max,373.000000


In [21]:
bins = [0, 20, 50, 104, 373]
labels = ["<=20", "21-50", "51-104", "105+"]
df['staleness_bucket'] = pd.cut(df['days_since_last_update'], bins=bins, labels=labels)
df['staleness_bucket'].value_counts()

,count
staleness_bucket,
<=20,15866
51-104,9080
21-50,4736
105+,318


In [23]:
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)
df['is_declining'].value_counts()

summary = df.groupby('staleness_bucket')['is_declining'].agg(['count', 'mean'])
print(summary)

                  count      mean
staleness_bucket                 
<=20              15866  0.538888
21-50              4736  0.421030
51-104             9080  0.610573
105+                318  0.547170


/tmp/ipykernel_2700/726302028.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  summary = df.groupby('staleness_bucket')['is_declining'].agg(['count', 'mean'])


## Signal Check #1 — Staleness (`days_since_last_update`)

**Bucket table:**

| Bucket | n | declining-rate |
|---|---|---|
| <=20 | 15,866 | 0.539 |
| 21-50 | 4,736 | 0.421 |
| 51-104 | 9,080 | 0.611 |
| 105+ | 318 | 0.547 |

**Verdict: MIXED**

The assumption was that staler content carries higher decline risk (a monotonic increase across buckets). The actual pattern is non-monotonic — the 21-50 day bucket has the lowest decline rate (0.421), followed by a jump to the highest rate in 51-104 days (0.611), then a drop again at 105+ (0.547). No consistent one-directional trend emerged, so staleness alone is not a reliable decline predictor in this bucketed form.

Note: the 105+ bucket has only n=318 (much smaller than the other buckets), so its estimate is noisier and shouldn't be weighted heavily.

In [28]:
df['impressions_90d'].describe()

,impressions_90d
count,30000.000000
mean,5200.366300
std,16838.019547
min,1.000000
25%,81.000000
50%,731.000000
75%,3615.250000
max,517715.000000


In [32]:
bins = [0, 81, 731, 3615, float("inf")]
labels = ["<=81", "82-731", "732-3615", "3615+"]
df['impression_bucket'] = pd.cut(df['impressions_90d'], bins=bins, labels=labels)
df['impression_bucket'].value_counts()

,count
impression_bucket,
<=81,7503
3615+,7500
82-731,7499
732-3615,7498


In [34]:
summary2 = df.groupby('impression_bucket')['is_declining'].agg(['count', 'mean'])
print(summary2)

                   count      mean
impression_bucket                 
<=81                7503  0.376116
82-731              7499  0.604614
732-3615            7498  0.625634
3615+               7500  0.562000


/tmp/ipykernel_2700/366428798.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  summary2 = df.groupby('impression_bucket')['is_declining'].agg(['count', 'mean'])


## Signal Check #2 — Visibility (`impressions_90d`)

**Bucket table:**

| Bucket | n | declining-rate |
|---|---|---|
| <=81 | 7,503 | 0.376 |
| 82-731 | 7,499 | 0.605 |
| 732-3615 | 7,498 | 0.626 |
| 3615+ | 7,500 | 0.562 |

**Verdict: CONFIRMED**

Higher visibility (impressions) correlates with a meaningfully higher decline rate — the lowest-visibility bucket declines at 37.6%, compared to 56-63% for all higher-visibility buckets. The direction holds across most of the range (3 of 4 buckets increase), with only a mild dip at the very top bucket. Bucket sizes are balanced (~7,500 each, from quartile-based bins), so no small-sample caveat applies here.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.